# CrisisGraph Qwen3-4B QLoRA — SFT V3

Train a fresh adapter from the same pinned base as V1, using 300 V3 training and 72 validation records. The exported files use the documented `pilot-first-pass` policy (AI-assisted review plus exact mechanical corrections); independent semantic acceptance remains pending.

Upload this notebook to a fresh Colab GPU runtime. Run top to bottom with `RUN_MODE = "smoke"`. After success, restart the runtime, set `RUN_MODE = "full"`, and run top to bottom again. Never continue training a V1 adapter or the smoke adapter.

The model extracts candidate facts only. Deterministic Rust services retain routing, verification, allocation, and dispatch authority. V3 remains an experiment until regression and blind-holdout evaluation.


## 1. Install the V1-recorded training stack

Versions below come from the saved full V1 run manifest. PyTorch/CUDA remain supplied by Colab to avoid replacing its GPU stack; any difference is reported and saved. If pinned versions cannot be installed, stop and record the compatibility issue rather than silently upgrading. Rerun installation after restarting the runtime if needed.


In [ ]:
%pip install -q "transformers==5.17.0" "trl==1.13.0" "peft==0.21.0" "accelerate==1.15.0" "bitsandbytes==0.50.2" "datasets==5.0.1" sentencepiece


In [ ]:
import gc
import hashlib
import importlib.metadata as metadata
import json
import os
import random
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU in Colab"

for package in ["torch", "transformers", "trl", "peft", "accelerate", "bitsandbytes", "datasets"]:
    print(f"{package}: {metadata.version(package)}")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA capability:", torch.cuda.get_device_capability(0))
V1_TORCH_VERSION = "2.11.0+cu128"
if torch.__version__ != V1_TORCH_VERSION:
    print(f"Runtime difference: V1 used torch {V1_TORCH_VERSION}; current runtime uses {torch.__version__}.")
    print("Record this difference when comparing the experiments.")


## 2. Configure the controlled experiment

The smoke run performs only three optimizer steps. It checks compatibility and memory use; it is not an accuracy result. The full run starts again from the untouched base model.

In [ ]:
RUN_MODE = "smoke"  # Change to "full" only after the smoke run passes.
assert RUN_MODE in {"smoke", "full"}

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_REVISION = "cdbee75f17c01a7cc42f958dc650907174af0554"
EXPERIMENT_ID = "qwen3-4b-qlora-sft-v3"
REVIEW_POLICY = "pilot-first-pass"
SEED = 42
MAX_LENGTH = 1024
OUTPUT_ROOT = Path("/content/crisisgraph-qlora-sft-v3")
OUTPUT_DIR = OUTPUT_ROOT / RUN_MODE

TRAIN_FILE = Path("/content/triage-sft-v3-train.jsonl")
VALIDATION_FILE = Path("/content/triage-sft-v3-validation.jsonl")
EXPECTED_SHA256 = {
    TRAIN_FILE.name: "4c6ffd57242df8e7d076914ab14015c0d8691db2a4783750c6a9152dfaf30db9",
    VALIDATION_FILE.name: "d22a5d10e02bda2297fd3d07b98ca7b5b3842dc7c989987dd9f3a731ef696351",
}

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise RuntimeError(f"{OUTPUT_DIR} already contains a run. Download it and choose a new OUTPUT_ROOT before proceeding.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Run mode:", RUN_MODE)
print("Output directory:", OUTPUT_DIR)

## 3. Upload and verify the exported datasets

Upload these two project files when prompted:

- `ml/data/generated/triage-sft-v3-train.jsonl`
- `ml/data/generated/triage-sft-v3-validation.jsonl`

Hash verification prevents accidentally training on an older candidate revision or on the frozen evaluation set.

In [ ]:
if not TRAIN_FILE.exists() or not VALIDATION_FILE.exists():
    from google.colab import files

    print("Upload the exported train and validation JSONL files.")
    uploaded = files.upload()
    for path in [TRAIN_FILE, VALIDATION_FILE]:
        if path.name in uploaded:
            path.write_bytes(uploaded[path.name])

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

for path in [TRAIN_FILE, VALIDATION_FILE]:
    assert path.exists(), f"Missing required upload: {path.name}"
    actual_hash = sha256(path)
    assert actual_hash == EXPECTED_SHA256[path.name], (
        f"Checksum mismatch for {path.name}: {actual_hash}"
    )
    print(path.name, actual_hash, "verified")

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": str(TRAIN_FILE),
        "validation": str(VALIDATION_FILE),
    },
)
assert len(dataset["train"]) == 300
assert len(dataset["validation"]) == 72

for split in ["train", "validation"]:
    for row in dataset[split]:
        assert [message["role"] for message in row["messages"]] == [
            "system", "user", "assistant"
        ]
        json.loads(row["messages"][2]["content"])
print(dataset)
print(json.dumps(dataset["train"][0], ensure_ascii=False, indent=2))

## 4. Load the 4-bit base model and inject LoRA adapters

The base weights stay frozen in NF4. Gradients update only the low-rank adapter matrices. This pilot uses FP16 compute across Colab GPU types and retains trainable adapter weights in FP32, avoiding BF16/AMP gradient-scaler incompatibilities across changing Colab package builds.

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

use_bf16 = False
compute_dtype = torch.float16

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    quantization_config=quantization_config,
    device_map={"": 0},
    torch_dtype=compute_dtype,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)
print("Compute dtype:", compute_dtype)

## 5. Configure supervised fine-tuning

Loss is computed only on assistant responses, so the model learns the target JSON instead of learning to reproduce the system prompt or SOS message. Packing is disabled for the first controlled run to keep example boundaries simple to inspect.

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    max_length=MAX_LENGTH,
    packing=False,
    assistant_only_loss=True,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=1e-4,
    num_train_epochs=2 if RUN_MODE == "full" else 1,
    max_steps=-1 if RUN_MODE == "full" else 3,
    warmup_steps=1 if RUN_MODE == "smoke" else 6,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    logging_steps=1 if RUN_MODE == "smoke" else 5,
    eval_strategy="epoch" if RUN_MODE == "full" else "no",
    save_strategy="epoch" if RUN_MODE == "full" else "no",
    load_best_model_at_end=RUN_MODE == "full",
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    fp16=not use_bf16,
    bf16=use_bf16,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"] if RUN_MODE == "full" else None,
    processing_class=tokenizer,
    peft_config=lora_config,
)

# Some Colab combinations initialize newly injected LoRA weights in the
# base compute dtype. AMP gradient scaling expects trainable master weights
# in FP32, so cast adapters only; the frozen base remains 4-bit NF4.
for parameter in trainer.model.parameters():
    if parameter.requires_grad and parameter.dtype != torch.float32:
        parameter.data = parameter.data.float()

trainable_dtypes = {parameter.dtype for parameter in trainer.model.parameters() if parameter.requires_grad}
assert trainable_dtypes == {torch.float32}, trainable_dtypes
print("Trainable parameter dtypes:", trainable_dtypes)
trainer.model.print_trainable_parameters()

In [ ]:
batch = next(iter(trainer.get_train_dataloader()))
supervised_tokens = int((batch["labels"] != -100).sum())
all_tokens = int(batch["labels"].numel())
assert supervised_tokens > 0
assert supervised_tokens < all_tokens
print({"supervised_assistant_tokens": supervised_tokens, "batch_tokens": all_tokens})

## 6. Train and validate

Do not treat smoke-run loss as an experiment result. For the full run, validation loss selects the best epoch checkpoint; task accuracy is measured later with the frozen 120-case CrisisGraph evaluator.

In [ ]:
train_result = trainer.train()
train_metrics = dict(train_result.metrics)
eval_metrics = trainer.evaluate() if RUN_MODE == "full" else {}
print("Train metrics:", json.dumps(train_metrics, indent=2, default=str))
print("Validation metrics:", json.dumps(eval_metrics, indent=2, default=str))

## 7. Save the adapter and run metadata

This saves only the PEFT adapter and tokenizer, not a second copy of the full base model. Keep the run manifest with the adapter so the result remains attributable to its data, model, and hyperparameters.

In [ ]:
ADAPTER_DIR = OUTPUT_DIR / "adapter"
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

run_manifest = {
    "experiment_id": EXPERIMENT_ID,
    "run_mode": RUN_MODE,
    "requested_base_revision": MODEL_REVISION,
    "gpu": torch.cuda.get_device_name(0),
    "cuda_version": torch.version.cuda,
    "base_model": MODEL_ID,
    "base_model_revision": getattr(trainer.model.config, "_commit_hash", None),
    "method": "QLoRA",
    "quantization": {
        "bits": 4,
        "type": "nf4",
        "double_quantization": True,
        "compute_dtype": str(compute_dtype),
    },
    "lora": {
        "rank": 16,
        "alpha": 32,
        "dropout": 0.05,
        "target_modules": "all-linear",
    },
    "training": {
        "seed": SEED,
        "max_length": MAX_LENGTH,
        "epochs_requested": 2 if RUN_MODE == "full" else 1,
        "max_steps": -1 if RUN_MODE == "full" else 3,
        "micro_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "learning_rate": 1e-4,
        "warmup_steps": 1 if RUN_MODE == "smoke" else 6,
        "assistant_only_loss": True,
    },
    "data": {
        "dataset_version": "triage-sft-v3",
        "review_policy": REVIEW_POLICY,
        "independent_review_complete": False,
        "prompt_version": "triage-extraction-v2.1.1",
        "dev_included": False,
        "train_records": len(dataset["train"]),
        "validation_records": len(dataset["validation"]),
        "sha256": EXPECTED_SHA256,
        "frozen_evaluation_included": False,
    },
    "metrics": {"train": train_metrics, "validation": eval_metrics},
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "packages": {
        package: metadata.version(package)
        for package in ["torch", "transformers", "trl", "peft", "accelerate", "bitsandbytes", "datasets"]
    },
}
(OUTPUT_DIR / "run-manifest.json").write_text(
    json.dumps(run_manifest, indent=2, default=str), encoding="utf-8"
)
print("Saved adapter to", ADAPTER_DIR)
print(json.dumps(run_manifest, indent=2, default=str))

## 8. Post-training generation smoke check

This checks that the trained adapter can generate a response. It is not the frozen evaluation and must not be reported as accuracy.

In [ ]:
system_prompt = dataset["train"][0]["messages"][0]["content"]
sample_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "18 people at Market Hall need a rescue boat. River Road is flooded for 3 hours."},
]
trained_model = trainer.model
trained_model.gradient_checkpointing_disable()
trained_model.config.use_cache = True
trained_model.eval()
inputs = tokenizer.apply_chat_template(
    sample_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(trained_model.device)
input_length = inputs["input_ids"].shape[-1]
with torch.inference_mode():
    generated = trained_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
response = tokenizer.decode(generated[0, input_length:], skip_special_tokens=True)
print("Adapter response:")
print(response)

with trained_model.disable_adapter(), torch.inference_mode():
    base_generated = trained_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
base_response = tokenizer.decode(
    base_generated[0, input_length:], skip_special_tokens=True
)
print("\nBase response with adapter disabled:")
print(base_response)

## 9. Download the V2 adapter and metadata

The ZIP includes the adapter, tokenizer, and run manifest together. Download after a successful full run. The next step is a base/V1/V2 comparison on the frozen regression suite with consistent inference settings, followed by a separately reviewed blind holdout for generalization claims. Conversion for local evaluation is not model promotion.


In [ ]:
import shutil

# Bundle the manifest with the adapter so the download remains attributable.
shutil.copy2(OUTPUT_DIR / "run-manifest.json", ADAPTER_DIR / "run-manifest.json")
archive = shutil.make_archive(
    str(OUTPUT_DIR / f"{EXPERIMENT_ID}-{RUN_MODE}-adapter"), "zip", ADAPTER_DIR
)
print("Created", archive)

from google.colab import files
files.download(archive)


## Experimental discipline

A successful training run only proves that optimization completed. The adapter is accepted only if the existing evaluator shows a meaningful improvement in human-review recall and unsupported-fact rate without unacceptable regressions in schema validity, clear-case accuracy, or multilingual behavior. The deterministic routing and verification boundary remains unchanged regardless of the model result.